In [3]:
pip install google-generativeai --no-deps

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [8]:
import os
import google.generativeai as genai

# 발급받은 API 키를 여기에 입력하세요
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]  # 키는 환경 변수로
genai.configure(api_key=GOOGLE_API_KEY)

# 모델 설정 (가장 성능이 좋은 Gemini 1.5 Pro 또는 Flash 추천)
model = genai.GenerativeModel('models/gemini-2.5-flash')

# 테스트 질문
response = model.generate_content("안녕 제미나이, 나를 도와줄 수 있어?")
print(response.text)

안녕하세요! 네, 물론이죠. 무엇을 도와드릴까요? 편하게 말씀해주세요!


In [11]:
import pandas as pd
import numpy as np
import google.generativeai as genai
from datetime import datetime
import time
import os

# 1. API 설정 (사용 가능한 모델 자동 선택)
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]  # 키는 환경 변수로
genai.configure(api_key=GOOGLE_API_KEY)

available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
target_model = available_models[0] if available_models else 'gemini-1.5-flash'
model = genai.GenerativeModel(target_model)

def get_persona_data(persona_desc, count):
    prompt = f"""
    로봇에게 '앞으로 가'라고 명령할 때 발생하는 데이터 셋을 만들고 있어.
    다음 페르소나의 특성에 맞는 명령어 {count}개를 생성해줘.
    페르소나 특성: {persona_desc}
    규칙: 1. 한 줄에 하나씩. 2. 설명/번호 없이 문구만. 3. 중복 최소화.
    """
    try:
        response = model.generate_content(prompt)
        return [l.strip() for l in response.text.strip().split('\n') if l.strip()][:count]
    except: return []

def verify_labels(commands, original_intent="앞으로 가"):
    """생성된 데이터의 라벨링이 의도대로 되었는지 AI가 재검토"""
    prompt = f"""
    다음 리스트의 명령어들이 '{original_intent}'와 의미상 동일한 동작을 지시하는지 판별해줘.
    동일하면 0, 다른 동작(방향 착각, 무관함 등)이면 1로 표시해줘.
    결과는 숫자만 콤마로 구분해서 순서대로 출력해. 예: 0,1,0,0...
    명령어 리스트: {commands}
    """
    try:
        response = model.generate_content(prompt)
        labels = [int(x.strip()) for x in response.text.strip().split(',')]
        return labels
    except: return [None] * len(commands)

print(f"✨ 시스템 가동 중... (연결 모델: {target_model})")

# 2. 목표치 설정 (4:6 비율 유지)
# 총 400개 중 라벨 0(정답군) 160개, 라벨 1(오답군) 240개 목표
target_0 = 160
target_1 = 240

# 데이터 수집
p1_raw = get_persona_data("스마트폰 오타 (앞으류 가, 아ㅍ으로 가 등)", 100)
p2_raw = get_persona_data("건망증 및 문장 변형 (가라 앞으로, 전방으로 가자 등)", 100)
p3_raw = get_persona_data("방향 인지 착각 (뒤로 가, 왼쪽 가, 옆으로 이동 등)", 150)
p4_raw = ["악으로 가", "압으로 가", "앙으로 가", "앞으로 갸", "앞으로 고"] # 박사님 리스트 (확장 가능)

# 3. 데이터 통합 및 1차 라벨링
temp_list = []
for c in p1_raw: temp_list.append({"명령어": c, "초기_라벨": 0, "페르소나": "P1_오타"})
for c in p2_raw: temp_list.append({"명령어": c, "초기_라벨": 0, "페르소나": "P2_건망증"})
for c in p3_raw: temp_list.append({"명령어": c, "초기_라벨": 1, "페르소나": "P3_방향착각"})
for c in p4_raw: temp_list.append({"명령어": c, "초기_라벨": 1, "페르소나": "P4_발음유사"})

df = pd.DataFrame(temp_list).drop_duplicates('명령어')

# 4. 라벨링 자가 검토 (정교함 추가)
print("🔍 AI가 라벨링의 적절성을 검토하고 있습니다...")
# API 부하를 줄이기 위해 20개씩 묶어서 검토
verified_labels = []
cmds = df['명령어'].tolist()
for i in range(0, len(cmds), 20):
    batch = cmds[i:i+20]
    verified_labels.extend(verify_labels(batch))
    time.sleep(1) # 속도 조절

df['최종_라벨링'] = verified_labels
# 검토 실패 시 초기 라벨 사용
df['최종_라벨링'] = df['최종_라벨링'].fillna(df['초기_라벨']).astype(int)

# 5. 샘플 수 및 비율 조정 (4:6 샘플링)
df_0 = df[df['최종_라벨링'] == 0].sample(n=min(len(df[df['최종_라벨링']==0]), target_0))
df_1 = df[df['최종_라벨링'] == 1].sample(n=min(len(df[df['최종_라벨링']==1]), target_1))

# 부족한 경우 추가 생성 대신 가용 자원 활용
final_df = pd.concat([df_0, df_1]).sample(frac=1).reset_index(drop=True)

# 6. 저장 및 보고
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
file_name = f"robot_data_balanced_400_{timestamp}.csv"
final_df.to_csv(file_name, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"📊 최종 데이터 생성 보고서")
print("="*50)
print(f"파일명: {file_name}")
print(f"총 샘플 수: {len(final_df)}개")
print(f"라벨 분포 (0:정답, 1:오답):\n{final_df['최종_라벨링'].value_counts(normalize=True).map(lambda x: f'{x:.1%}')}")
print("-" * 30)
print(f"페르소나별 분포:\n{final_df['페르소나'].value_counts()}")
print("="*50)

✨ 시스템 가동 중... (연결 모델: models/gemini-2.5-flash)
🔍 AI가 라벨링의 적절성을 검토하고 있습니다...

📊 최종 데이터 생성 보고서
파일명: robot_data_balanced_400_20260216_173725.csv
총 샘플 수: 310개
라벨 분포 (0:정답, 1:오답):
최종_라벨링
0    51.6%
1    48.4%
Name: proportion, dtype: object
------------------------------
페르소나별 분포:
페르소나
P3_방향착각    141
P2_건망증      86
P1_오타       79
P4_발음유사      4
Name: count, dtype: int64


In [16]:
import os
import pandas as pd
import numpy as np
import google.generativeai as genai
from datetime import datetime
import time

# API 설정
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]  # 키는 환경 변수로
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

def get_persona_data(persona_desc, count):
    prompt = f"""
    로봇에게 '앞으로 가'라고 명령하는 짧은 문장 {count}개를 생성해줘.
    특성: {persona_desc}
    조건: 한 줄에 하나씩, 번호나 기호 없이 문장만 출력해.
    """
    try:
        response = model.generate_content(prompt)
        # 텍스트 정제 및 빈 줄 제거
        lines = [l.strip() for l in response.text.strip().split('\n') if len(l.strip()) > 1]
        return lines[:count]
    except Exception as e:
        print(f"데이터 생성 오류: {e}")
        return []

def verify_labels(commands):
    if not commands: return []
    prompt = f"""
    다음 문장들이 '앞으로 가(전진)'의미면 0, 아니면(후진, 정지, 무관 등) 1로 분류해.
    결과는 숫자만 콤마로 구분해서 출력해. 예: 0,1,0...
    문장: {", ".join(commands)}
    """
    try:
        response = model.generate_content(prompt)
        import re
        # 숫자만 찾아내기
        labels = [int(s) for s in re.findall(r'[01]', response.text)]
        return labels
    except:
        return [0] * len(commands) # 실패 시 기본값 0

def create_balanced_set(persona_name, desc, file_prefix):
    print(f"🔄 {persona_name} 데이터 생성 중...")
    # 비율을 맞추기 위해 넉넉하게 150개 요청
    raw_cmds = get_persona_data(desc, 150)
    
    if not raw_cmds:
        print(f"⚠️ {persona_name} 데이터를 가져오지 못했습니다.")
        return None, 0

    # 라벨링 검토
    labels = []
    for i in range(0, len(raw_cmds), 30):
        batch = raw_cmds[i:i+30]
        labels.extend(verify_labels(batch))
        time.sleep(0.5)

    df = pd.DataFrame({"명령어": raw_cmds, "최종_라벨링": labels[:len(raw_cmds)], "페르소나": persona_name})
    
    # 0과 1 분리 (약 50:50 목표)
    df_0 = df[df['최종_라벨링'] == 0]
    df_1 = df[df['최종_라벨링'] == 1]
    
    # 최소 50개씩 확보하되, 부족하면 있는 만큼 다 사용
    final_df = pd.concat([
        df_0.head(50), 
        df_1.head(50)
    ]).sample(frac=1).reset_index(drop=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    file_name = f"{file_prefix}_{timestamp}.csv"
    final_df.to_csv(file_name, index=False, encoding='utf-8-sig')
    return file_name, len(final_df)

# --- 실행부 ---

# P5: 현대 한국어 (외래어, 줄임말)
p5_desc = "현대 한국인의 외래어(Go, 무브, 스트레이트), 줄임말(앞가, 전꼬), 급한 명령어가 포함된 전진 및 오답 데이터"
v_file, v_len = create_balanced_set("P5_Modern", p5_desc, "valid_set")

# P6: P1~P5 35% + 신규 65% (실전 상황)
p6_desc = "오타, 문장변형, 방향오류, 외래어가 섞인 명령어 35%와 완전히 새로운 사투리나 복합 명령 65%가 섞인 실전 데이터"
t_file, t_len = create_balanced_set("P6_Test", p6_desc, "test_set")

print("\n" + "="*50)
print(f"✅ 생성 완료 보고서")
if v_file: print(f"1. 검증 세트: {v_file} ({v_len}개)")
if t_file: print(f"2. 테스트 세트: {t_file} ({t_len}개)")
print("="*50)

🔄 P5_Modern 데이터 생성 중...
🔄 P6_Test 데이터 생성 중...

✅ 생성 완료 보고서
1. 검증 세트: valid_set_20260216_233352.csv (73개)
2. 테스트 세트: test_set_20260216_233524.csv (56개)
